# Structure of the disc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "oph_table_reveal.gif"

FPS = 24
DURATION_SEC = 10
FRAMES = FPS * DURATION_SEC

FIGSIZE = (14, 6)
DPI = 120

BG = "#020510"
TEXT = "#f0f6ff"
MUTED = "#6f8194"
LINE = "#2b4358"

# =========================
# Data
# =========================

headers = [
    "λ (μm)", "dₙₑb (″)", "FRint (T/B)",
    "RFWHM B (″)", "RFWHM T (″)",
    "RFW10% B (″)", "RFW10% T (″)"
]

rows = [
    ["0.6",  "0.42 ± 0.01", "3.0", "0.93 ± 0.11", "0.82 ± 0.11", "1.88 ± 0.08", "1.86 ± 0.05"],
    ["0.8",  "0.40 ± 0.01", "3.3", "0.99 ± 0.12", "0.92 ± 0.12", "2.00 ± 0.04", "1.94 ± 0.04"],
    ["2.00", "0.34 ± 0.01", "3.5", "1.12 ± 0.10", "0.76 ± 0.07", "2.17 ± 0.03", "1.83 ± 0.07"],
    ["4.44", "0.21 ± 0.01", "3.8", "0.71 ± 0.07", "0.62 ± 0.12", "1.84 ± 0.07", "1.33 ± 0.08"],
]

# =========================
# Figure
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

# =========================
# Helpers
# =========================

def smoothstep(a, b, x):
    t = np.clip((x-a)/(b-a), 0, 1)
    return t*t*(3-2*t)

# =========================
# Layout
# =========================

x_positions = np.linspace(0.08, 0.92, len(headers))
y_header = 0.75
row_gap = 0.12

# Header text
header_artists = []
for i, h in enumerate(headers):
    txt = ax.text(
        x_positions[i], y_header,
        h,
        color=TEXT,
        fontsize=14,
        ha="center",
        va="center",
        alpha=0
    )
    header_artists.append(txt)

# Lines
line_top = ax.plot([0.05, 0.95], [0.82, 0.82], color=LINE, alpha=0)[0]
line_mid = ax.plot([0.05, 0.95], [0.68, 0.68], color=LINE, alpha=0)[0]
line_bottom = ax.plot([0.05, 0.95], [0.15, 0.15], color=LINE, alpha=0)[0]

# Rows
row_artists = []

for r, row in enumerate(rows):
    y = y_header - (r+1)*row_gap
    row_line = []

    for c, val in enumerate(row):
        txt = ax.text(
            x_positions[c], y,
            val,
            color=TEXT,
            fontsize=13,
            ha="center",
            va="center",
            alpha=0
        )
        row_line.append(txt)

    row_artists.append(row_line)

# Title
title = ax.text(
    0.05, 0.92,
    "Oph163131 Morphological Properties",
    color=TEXT,
    fontsize=18,
    ha="left",
    alpha=0
)

# =========================
# Animation
# =========================

def update(frame):
    t = frame / (FRAMES - 1)

    a_title = smoothstep(0.02, 0.12, t)
    a_header = smoothstep(0.10, 0.20, t)
    a_lines = smoothstep(0.18, 0.28, t)

    title.set_alpha(a_title)

    for h in header_artists:
        h.set_alpha(a_header)

    line_top.set_alpha(a_lines)
    line_mid.set_alpha(a_lines)
    line_bottom.set_alpha(a_lines)

    # rows appear sequentially
    for i, row in enumerate(row_artists):
        a = smoothstep(0.25 + i*0.12, 0.35 + i*0.12, t)

        for cell in row:
            cell.set_alpha(a)

    return (
        [title, line_top, line_mid, line_bottom]
        + header_artists
        + [cell for row in row_artists for cell in row]
    )

# =========================
# Save
# =========================

anim = FuncAnimation(fig, update, frames=FRAMES)

anim.save(GIF_PATH, writer=PillowWriter(fps=FPS))

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print("Saved:", GIF_PATH)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "oph163131_profile_visibility_dark.gif"

FPS = 24
DURATION_SEC = 12
FRAMES = FPS * DURATION_SEC

FIGSIZE = (9, 11)
DPI = 120

BG = "#020510"
GRID = "#2b4358"
TEXT = "#c9d3df"
MUTED = "#6f8194"

ORANGE = "#ff9a3c"
BLUE = "#3d6cff"
RED = "#ff5a45"
WHITE = "#f0f6ff"
CYAN = "#35c9ff"

rng = np.random.default_rng(163131)

# =========================
# Top panel data: normalized intensity profile
# =========================

x = np.linspace(-1.45, 1.45, 900)

def gauss(x, mu, amp, sig):
    return amp * np.exp(-0.5 * ((x - mu) / sig) ** 2)

profile_fit = (
    0.03
    + gauss(x, 0.00, 1.05, 0.045)
    + gauss(x, -0.48, 0.27, 0.18)
    + gauss(x, 0.48, 0.27, 0.18)
    + gauss(x, -0.78, 0.13, 0.10)
    + gauss(x, 0.78, 0.13, 0.10)
    - gauss(x, -0.35, 0.10, 0.055)
    - gauss(x, 0.35, 0.10, 0.055)
    - gauss(x, -0.63, 0.11, 0.040)
    - gauss(x, 0.63, 0.11, 0.040)
)

profile_fit = np.clip(profile_fit, 0, None)
profile_cut = profile_fit * 0.82 + rng.normal(0, 0.035, x.size)
profile_cut += 0.03 * np.sin(19 * x) + 0.02 * np.sin(45 * x)
profile_cut = np.clip(profile_cut, -0.08, None)

uncert = 0.045 + 0.020 * np.exp(-x**2 / 0.4)

# top axis radius conversion: approximately 1 arcsec ≈ 120 au
radius_au_ticks = np.array([-150, -75, 0, 75, 150])
offset_ticks_for_radius = radius_au_ticks / 120.0

# =========================
# Bottom panel data: visibility profile
# =========================

baseline = np.logspace(3.6, 7.25, 1800)
logb = np.log10(baseline)

vis_fit = (
    48 / (1 + np.exp((logb - 4.95) / 0.16))
    - 5.0 * np.exp(-0.5 * ((logb - 5.22) / 0.12) ** 2)
    + 2.3 * np.exp(-0.5 * ((logb - 5.58) / 0.09) ** 2)
    + 2.8 * np.exp(-0.5 * ((logb - 5.78) / 0.055) ** 2)
    + 1.1 * np.exp(-0.5 * ((logb - 6.05) / 0.08) ** 2)
)

noise_amp = 0.6 + 18 * np.clip((logb - 6.3) / 0.9, 0, None) ** 2
vis_data = vis_fit + rng.normal(0, noise_amp, baseline.size)

# sparse red fit points, black dense data
fit_sample = np.linspace(0, baseline.size - 1, 260).astype(int)

# =========================
# Figure setup
# =========================

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=FIGSIZE,
    dpi=DPI,
    gridspec_kw={"height_ratios": [1, 1], "hspace": 0.20}
)

fig.patch.set_facecolor(BG)

for ax in (ax1, ax2):
    ax.set_facecolor(BG)
    ax.grid(True, color=GRID, linestyle=":", linewidth=0.8, alpha=0.55)
    ax.tick_params(colors=TEXT, labelsize=11)

    for spine in ax.spines.values():
        spine.set_color(GRID)
        spine.set_linewidth(1.2)

# Top panel
ax1.set_xlim(-1.45, 1.45)
ax1.set_ylim(-0.18, 1.35)
ax1.set_xlabel("Offset (arcsec)", color=TEXT, fontsize=13)
ax1.set_ylabel("Normalized intensity", color=TEXT, fontsize=13)

ax1_top = ax1.secondary_xaxis(
    "top",
    functions=(lambda off: off * 120.0, lambda au: au / 120.0)
)
ax1_top.set_xlabel("Radius (au)", color=TEXT, fontsize=13)
ax1_top.set_xticks(radius_au_ticks)
ax1_top.tick_params(colors=TEXT, labelsize=11)

# Bottom panel
ax2.set_xscale("log")
ax2.set_xlim(baseline.min(), baseline.max())
ax2.set_ylim(-42, 52)
ax2.set_xlabel(r"Baseline ($\lambda$)", color=TEXT, fontsize=13)
ax2.set_ylabel("Re(V) (mJy)", color=TEXT, fontsize=13)
ax2.axhline(0, color=MUTED, linestyle=(0, (5, 4)), linewidth=1.3, alpha=0.75)

# Title
fig.text(
    0.5,
    0.965,
    "Oph163131 — DISC PROFILE AND VISIBILITY FIT",
    color=TEXT,
    fontsize=18,
    fontweight="bold",
    ha="center",
    va="center"
)

fig.text(
    0.5,
    0.935,
    "illustrative reconstruction • major-axis cut • Fourier visibility fit",
    color=MUTED,
    fontsize=11,
    ha="center",
    va="center"
)

# =========================
# Static annotations top
# =========================

ax1.text(
    -0.98,
    1.12,
    "Beam",
    color=TEXT,
    fontsize=10,
    fontweight="bold"
)
ax1.plot([-0.87, -0.82], [1.04, 1.04], color=TEXT, linewidth=2.0)

ann_specs = [
    (-0.22, 0.42, "Small peak"),
    (-0.40, 0.33, "Ring 1"),
    (-0.64, 0.22, "Ring 2"),
    (-0.58, 0.09, "Gap 1"),
    (0.30, 0.13, "Region 2"),
]

annotation_artists = []

for xx, yy, label in ann_specs:
    txt = ax1.text(
        xx,
        yy + 0.10,
        label,
        color=TEXT,
        fontsize=9,
        fontweight="bold",
        alpha=0.0
    )
    marker = ax1.scatter(
        [xx],
        [yy],
        s=32,
        color=CYAN,
        edgecolor=WHITE,
        linewidths=0.5,
        alpha=0.0,
        zorder=20
    )
    annotation_artists.append((txt, marker))

# =========================
# Animated artists
# =========================

# Top panel
cut_fill = ax1.fill_between([], [], [], color=ORANGE, alpha=0.0)

cut_glow, = ax1.plot([], [], color=ORANGE, linewidth=8, alpha=0.0)
cut_line, = ax1.plot([], [], color=ORANGE, linewidth=2.0, label="Major axis cut")

fit_glow, = ax1.plot([], [], color=BLUE, linewidth=8, alpha=0.0)
fit_line, = ax1.plot([], [], color=BLUE, linewidth=2.2, label="Frank fit ×1.2")

cursor_top = ax1.scatter([], [], s=45, color=WHITE, edgecolor=CYAN, linewidths=0.8, alpha=0.0, zorder=30)

leg1 = ax1.legend(
    loc="upper right",
    facecolor="#07111f",
    edgecolor=GRID,
    labelcolor=TEXT,
    fontsize=10
)

for txt in leg1.get_texts():
    txt.set_color(TEXT)

# Bottom panel
data_scatter = ax2.scatter(
    [],
    [],
    s=4,
    color=WHITE,
    alpha=0.42,
    linewidths=0,
    label="Data",
    zorder=8
)

fit_scatter = ax2.scatter(
    [],
    [],
    s=4,
    color=RED,
    alpha=0.95,
    linewidths=0,
    label="Frank fit",
    zorder=10
)

fit_bottom_line, = ax2.plot([], [], color=RED, linewidth=1.5, alpha=0.85)

cursor_bottom = ax2.scatter([], [], s=34, color=RED, edgecolor=WHITE, linewidths=0.5, alpha=0.0, zorder=20)

leg2 = ax2.legend(
    loc="lower left",
    facecolor="#07111f",
    edgecolor=GRID,
    labelcolor=TEXT,
    fontsize=10
)

for txt in leg2.get_texts():
    txt.set_color(TEXT)

status = fig.text(
    0.08,
    0.035,
    "stage: scanning major-axis intensity profile",
    color=CYAN,
    fontsize=11,
    ha="left",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor=GRID,
        alpha=0.90
    )
)

# =========================
# Animation helpers
# =========================

def smoothstep(a, b, v):
    t = np.clip((v - a) / (b - a), 0, 1)
    return t * t * (3 - 2 * t)

def ease(v):
    return 1 - (1 - v) ** 3

# =========================
# Animation update
# =========================

def update(frame):
    global cut_fill

    t = frame / (FRAMES - 1)

    # Top panel draw left-to-right
    p_top = ease(smoothstep(0.02, 0.48, t))
    xmax = x.min() + p_top * (x.max() - x.min())
    m_top = x <= xmax

    xv = x[m_top]
    y_cut = profile_cut[m_top]
    y_fit = profile_fit[m_top]

    cut_line.set_data(xv, y_cut)
    cut_glow.set_data(xv, y_cut)
    cut_glow.set_alpha(0.08 * smoothstep(0.02, 0.15, t))

    fit_line.set_data(xv, y_fit)
    fit_glow.set_data(xv, y_fit)
    fit_glow.set_alpha(0.10 * smoothstep(0.14, 0.36, t))

    cut_fill.remove()
    if len(xv) > 0:
        cut_fill = ax1.fill_between(
            xv,
            y_cut - uncert[m_top],
            y_cut + uncert[m_top],
            color=ORANGE,
            alpha=0.18 * smoothstep(0.03, 0.18, t)
        )

        cursor_top.set_offsets([[xv[-1], y_cut[-1]]])
        cursor_top.set_alpha(0.80 * smoothstep(0.04, 0.20, t))

    # Annotations appear after profile is visible
    for i, (txt, marker) in enumerate(annotation_artists):
        a = smoothstep(0.48 + i * 0.045, 0.58 + i * 0.045, t)
        txt.set_alpha(0.90 * a)
        marker.set_alpha(0.75 * a)

    # Bottom panel draw over time
    p_bottom = ease(smoothstep(0.42, 0.90, t))
    log_min, log_max = np.log10(baseline.min()), np.log10(baseline.max())
    current_log = log_min + p_bottom * (log_max - log_min)
    m_bottom = logb <= current_log

    bx = baseline[m_bottom]
    by = vis_data[m_bottom]

    data_scatter.set_offsets(np.column_stack([bx, by]))

    fit_mask = m_bottom[fit_sample]
    fit_idx = fit_sample[fit_mask]

    fit_scatter.set_offsets(np.column_stack([baseline[fit_idx], vis_fit[fit_idx]]))

    fit_bottom_line.set_data(baseline[m_bottom], vis_fit[m_bottom])
    fit_bottom_line.set_alpha(0.85 * smoothstep(0.55, 0.75, t))

    if len(bx) > 0:
        cursor_bottom.set_offsets([[bx[-1], np.interp(bx[-1], baseline, vis_fit)]])
        cursor_bottom.set_alpha(0.8 * smoothstep(0.48, 0.65, t))

    if t < 0.45:
        status.set_text("stage: scanning major-axis intensity profile")
        status.set_color(CYAN)
    elif t < 0.70:
        status.set_text("stage: profile features identified — rings, gap, small peak")
        status.set_color(ORANGE)
    else:
        status.set_text("stage: visibility data compared with model fit")
        status.set_color(RED)

    return (
        cut_line,
        cut_glow,
        fit_line,
        fit_glow,
        cursor_top,
        data_scatter,
        fit_scatter,
        fit_bottom_line,
        cursor_bottom,
        status,
        cut_fill,
        *[a for pair in annotation_artists for a in pair]
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "sed_model_data_dark.gif"

FPS = 24
DURATION_SEC = 10
FRAMES = FPS * DURATION_SEC

FIGSIZE = (9, 6)
DPI = 130

BG = "#020510"
GRID = "#2b4358"
TEXT = "#c9d3df"
MUTED = "#6f8194"

RED = "#ff5a45"
BLUE = "#3d6cff"
CYAN = "#35c9ff"
WHITE = "#f0f6ff"

rng = np.random.default_rng(104)

# =========================
# Synthetic SED data
# =========================

lam = np.logspace(np.log10(0.40), np.log10(2600), 900)  # μm
log_lam = np.log10(lam)

def log_bump(logx, center, height, width):
    return height * np.exp(-0.5 * ((logx - center) / width) ** 2)

# Work in log10(nuFnu)
log_flux = (
    -17.8
    + log_bump(log_lam, 0.05, 3.95, 0.36)     # stellar / scattered light bump
    + log_bump(log_lam, 1.85, 4.25, 0.46)     # dust thermal bump
    - log_bump(log_lam, 1.02, 1.20, 0.18)     # mid-IR valley
)

# Far-IR/mm decline
log_flux -= 0.52 * np.clip(log_lam - 2.25, 0, None) ** 2.1

model = 10 ** log_flux

# Observational points
data_lam = np.array([
    0.35, 0.45, 0.55, 0.62, 0.70, 0.80, 0.92, 1.05, 1.20, 1.40,
    1.65, 2.10, 2.60, 3.20, 4.00, 5.00, 7.00, 10.0, 14.0,
    22.0, 30.0, 45.0, 65.0, 85.0, 110.0, 160.0, 250.0, 350.0,
    500.0, 850.0, 1300.0, 2500.0
])

data_model = np.interp(np.log10(data_lam), log_lam, np.log10(model))
data_flux = 10 ** (data_model + rng.normal(0, 0.14, size=data_lam.size))

# one strong errorbar point at the end, like reference
data_yerr = data_flux * 0.0
data_yerr[-1] = data_flux[-1] * 0.65

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0.14, right=0.96, top=0.88, bottom=0.14)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(0.25, 4000)
ax.set_ylim(1e-18, 1e-13)

ax.grid(True, which="both", color=GRID, linestyle=":", linewidth=0.75, alpha=0.55)

for spine in ax.spines.values():
    spine.set_color(GRID)
    spine.set_linewidth(1.2)

ax.tick_params(colors=TEXT, labelsize=11, which="both")

ax.set_xlabel(r"$\lambda$  ($\mu$m)", color=TEXT, fontsize=14, labelpad=10)
ax.set_ylabel(r"$\nu F_{\nu}$  (W m$^{-2}$)", color=TEXT, fontsize=14, labelpad=10)

fig.text(
    0.5,
    0.945,
    "SPECTRAL ENERGY DISTRIBUTION",
    color=TEXT,
    fontsize=18,
    fontweight="bold",
    ha="center",
    va="center"
)

fig.text(
    0.5,
    0.905,
    "model curve and photometric data points",
    color=MUTED,
    fontsize=11,
    ha="center",
    va="center"
)

# =========================
# Animated artists
# =========================

model_glow, = ax.plot([], [], color=RED, linewidth=7, alpha=0.08, zorder=8)
model_line, = ax.plot([], [], color=RED, linewidth=2.1, label="Model", zorder=9)

data_scatter = ax.scatter(
    [],
    [],
    s=34,
    color=BLUE,
    edgecolor=CYAN,
    linewidths=0.4,
    alpha=0.0,
    label="Data",
    zorder=12
)

# errorbar drawn manually for stable animation
err_line, = ax.plot([], [], color=BLUE, linewidth=1.2, alpha=0.0, zorder=11)
err_cap1, = ax.plot([], [], color=BLUE, linewidth=1.2, alpha=0.0, zorder=11)
err_cap2, = ax.plot([], [], color=BLUE, linewidth=1.2, alpha=0.0, zorder=11)

cursor = ax.scatter(
    [],
    [],
    s=45,
    color=WHITE,
    edgecolor=RED,
    linewidths=0.7,
    alpha=0.0,
    zorder=15
)

legend = ax.legend(
    loc="lower left",
    facecolor="#07111f",
    edgecolor=GRID,
    labelcolor=TEXT,
    fontsize=11
)

for txt in legend.get_texts():
    txt.set_color(TEXT)

status = ax.text(
    0.04,
    0.94,
    "stage: drawing SED model",
    transform=ax.transAxes,
    color=RED,
    fontsize=11,
    ha="left",
    va="top",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor=GRID,
        alpha=0.88
    )
)

# =========================
# Helpers
# =========================

def smoothstep(a, b, v):
    t = np.clip((v - a) / (b - a), 0, 1)
    return t * t * (3 - 2 * t)

def ease(v):
    return 1 - (1 - v) ** 3

# =========================
# Animation
# =========================

def update(frame):
    t = frame / (FRAMES - 1)

    p_model = ease(smoothstep(0.03, 0.58, t))
    log_min = np.log10(lam.min())
    log_max = np.log10(lam.max())
    current_log = log_min + p_model * (log_max - log_min)

    mask = log_lam <= current_log

    model_line.set_data(lam[mask], model[mask])
    model_glow.set_data(lam[mask], model[mask])

    model_glow.set_alpha(0.09 * smoothstep(0.05, 0.20, t))

    if np.any(mask):
        xcur = lam[mask][-1]
        ycur = model[mask][-1]
        cursor.set_offsets([[xcur, ycur]])
        cursor.set_alpha(0.85 * smoothstep(0.06, 0.18, t))

    # Data points reveal after model curve starts
    data_reveal = smoothstep(0.34, 0.80, t)
    n = int(data_reveal * len(data_lam))

    if n > 0:
        data_scatter.set_offsets(np.column_stack([data_lam[:n], data_flux[:n]]))
        data_scatter.set_alpha(0.90)

    # Manual last errorbar appears near the end
    err_a = smoothstep(0.78, 0.92, t)

    xl = data_lam[-1]
    yl = data_flux[-1]
    ye = data_yerr[-1]

    err_line.set_data([xl, xl], [yl - ye, yl + ye])
    err_cap1.set_data([xl * 0.92, xl * 1.08], [yl - ye, yl - ye])
    err_cap2.set_data([xl * 0.92, xl * 1.08], [yl + ye, yl + ye])

    err_line.set_alpha(0.9 * err_a)
    err_cap1.set_alpha(0.9 * err_a)
    err_cap2.set_alpha(0.9 * err_a)

    if t < 0.45:
        status.set_text("stage: drawing SED model")
        status.set_color(RED)
    elif t < 0.80:
        status.set_text("stage: photometric data appear")
        status.set_color(BLUE)
    else:
        status.set_text("stage: far-IR/mm uncertainty highlighted")
        status.set_color(CYAN)

    return (
        model_line,
        model_glow,
        data_scatter,
        err_line,
        err_cap1,
        err_cap2,
        cursor,
        status,
    )

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "oph163131_table2_fwhm_reveal.gif"

FPS = 24
DURATION_SEC = 8
FRAMES = FPS * DURATION_SEC

FIGSIZE = (14, 6)
DPI = 120

BG = "#020510"
TEXT = "#f0f6ff"
MUTED = "#6f8194"
LINE = "#2b4358"
CYAN = "#35c9ff"
ORANGE = "#ff9a3c"

# =========================
# Data
# =========================

title = "Table 2. FWHM of the Central Source at MIRI Wavelengths"

headers = ["λ", "Observed", "WebbPSF Model", "Deconvolved"]
subheaders = ["(μm)", "(″)", "(″)", "(″), (au)"]

rows = [
    ["7.7",  "0.44", "0.24", "0.37, 54"],
    ["12.8", "0.65", "0.45", "0.47, 70"],
    ["21.0", "1.09", "0.71", "0.83, 122"],
]

# =========================
# Figure
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

# =========================
# Helpers
# =========================

def smoothstep(a, b, x):
    t = np.clip((x - a) / (b - a), 0, 1)
    return t * t * (3 - 2 * t)

# =========================
# Layout
# =========================

x = [0.22, 0.40, 0.58, 0.77]

y_title = 0.90
y_header = 0.73
y_subheader = 0.60
y_rows = [0.46, 0.32, 0.18]

title_artist = ax.text(
    0.06, y_title,
    title,
    color=TEXT,
    fontsize=18,
    fontweight="bold",
    ha="left",
    va="center",
    alpha=0
)

lines = [
    ax.plot([0.18, 0.84], [0.82, 0.82], color=LINE, linewidth=1.3, alpha=0)[0],
    ax.plot([0.18, 0.84], [0.67, 0.67], color=LINE, linewidth=1.3, alpha=0)[0],
    ax.plot([0.18, 0.84], [0.53, 0.53], color=LINE, linewidth=1.3, alpha=0)[0],
    ax.plot([0.18, 0.84], [0.10, 0.10], color=LINE, linewidth=1.3, alpha=0)[0],
]

header_artists = []
for i, h in enumerate(headers):
    txt = ax.text(
        x[i], y_header,
        h,
        color=ORANGE if i == 3 else TEXT,
        fontsize=16,
        ha="center",
        va="center",
        alpha=0
    )
    header_artists.append(txt)

subheader_artists = []
for i, h in enumerate(subheaders):
    txt = ax.text(
        x[i], y_subheader,
        h,
        color=MUTED,
        fontsize=14,
        ha="center",
        va="center",
        alpha=0
    )
    subheader_artists.append(txt)

row_artists = []

for r, row in enumerate(rows):
    row_cells = []
    for c, val in enumerate(row):
        txt = ax.text(
            x[c], y_rows[r],
            val,
            color=CYAN if c == 3 else TEXT,
            fontsize=15,
            ha="center",
            va="center",
            alpha=0
        )
        row_cells.append(txt)
    row_artists.append(row_cells)

status = ax.text(
    0.06, 0.055,
    "revealing MIRI FWHM measurements",
    color=MUTED,
    fontsize=11,
    ha="left",
    va="center",
    alpha=0
)

# =========================
# Animation
# =========================

def update(frame):
    t = frame / (FRAMES - 1)

    a_title = smoothstep(0.02, 0.12, t)
    a_header = smoothstep(0.10, 0.22, t)
    a_lines = smoothstep(0.14, 0.26, t)

    title_artist.set_alpha(a_title)
    status.set_alpha(a_title)

    for line in lines:
        line.set_alpha(0.95 * a_lines)

    for txt in header_artists:
        txt.set_alpha(a_header)

    for txt in subheader_artists:
        txt.set_alpha(a_header)

    for i, row in enumerate(row_artists):
        a_row = smoothstep(0.30 + i * 0.17, 0.44 + i * 0.17, t)
        for cell in row:
            cell.set_alpha(a_row)

    return (
        [title_artist, status]
        + lines
        + header_artists
        + subheader_artists
        + [cell for row in row_artists for cell in row]
    )

# =========================
# Save
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

anim.save(GIF_PATH, writer=PillowWriter(fps=FPS))

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "oph163131_table3_photometry_reveal.gif"

FPS = 24
DURATION_SEC = 9
FRAMES = FPS * DURATION_SEC

FIGSIZE = (14, 6)
DPI = 120

BG = "#020510"
TEXT = "#f0f6ff"
MUTED = "#6f8194"
LINE = "#2b4358"
CYAN = "#35c9ff"
ORANGE = "#ff9a3c"

# =========================
# Data
# =========================

title = "Table 3. Photometry of Oph163131"

headers = ["λ", "F₇″.₅×₄″.₅", "F₂″×₂″"]
subheaders = ["(μm)", "(mJy)", "(mJy)"]

rows = [
    ["2.0",  "6.5 ± 0.2",  "…"],
    ["4.44", "2.5 ± 0.1",  "…"],
    ["7.70", "2.9 ± 0.1",  "1.6 ± 0.1"],
    ["12.8", "3.2 ± 0.1",  "1.9 ± 0.1"],
    ["21.0", "35.2 ± 1.1", "28.5 ± 0.9"],
]

# =========================
# Figure
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

# =========================
# Helpers
# =========================

def smoothstep(a, b, x):
    t = np.clip((x - a) / (b - a), 0, 1)
    return t * t * (3 - 2 * t)

# =========================
# Layout
# =========================

x = [0.36, 0.52, 0.68]

y_title = 0.90
y_header = 0.74
y_subheader = 0.61
y_rows = [0.48, 0.37, 0.26, 0.15, 0.04]

title_artist = ax.text(
    0.06, y_title,
    title,
    color=TEXT,
    fontsize=18,
    fontweight="bold",
    ha="left",
    va="center",
    alpha=0
)

lines = [
    ax.plot([0.31, 0.73], [0.82, 0.82], color=LINE, linewidth=1.3, alpha=0)[0],
    ax.plot([0.31, 0.73], [0.68, 0.68], color=LINE, linewidth=1.3, alpha=0)[0],
    ax.plot([0.31, 0.73], [0.54, 0.54], color=LINE, linewidth=1.3, alpha=0)[0],
    ax.plot([0.31, 0.73], [-0.025, -0.025], color=LINE, linewidth=1.3, alpha=0)[0],
]

header_artists = []
for i, h in enumerate(headers):
    txt = ax.text(
        x[i], y_header,
        h,
        color=ORANGE if i == 1 else TEXT,
        fontsize=16,
        ha="center",
        va="center",
        alpha=0
    )
    header_artists.append(txt)

subheader_artists = []
for i, h in enumerate(subheaders):
    txt = ax.text(
        x[i], y_subheader,
        h,
        color=MUTED,
        fontsize=14,
        ha="center",
        va="center",
        alpha=0
    )
    subheader_artists.append(txt)

row_artists = []

for r, row in enumerate(rows):
    row_cells = []
    for c, val in enumerate(row):
        txt = ax.text(
            x[c], y_rows[r],
            val,
            color=CYAN if c == 1 else TEXT,
            fontsize=15,
            ha="center",
            va="center",
            alpha=0
        )
        row_cells.append(txt)
    row_artists.append(row_cells)

status = ax.text(
    0.06, 0.055,
    "revealing photometry measurements",
    color=MUTED,
    fontsize=11,
    ha="left",
    va="center",
    alpha=0
)

# =========================
# Animation
# =========================

def update(frame):
    t = frame / (FRAMES - 1)

    a_title = smoothstep(0.02, 0.12, t)
    a_header = smoothstep(0.10, 0.22, t)
    a_lines = smoothstep(0.14, 0.26, t)

    title_artist.set_alpha(a_title)
    status.set_alpha(a_title)

    for line in lines:
        line.set_alpha(0.95 * a_lines)

    for txt in header_artists:
        txt.set_alpha(a_header)

    for txt in subheader_artists:
        txt.set_alpha(a_header)

    for i, row in enumerate(row_artists):
        a_row = smoothstep(0.28 + i * 0.12, 0.40 + i * 0.12, t)
        for cell in row:
            cell.set_alpha(a_row)

    return (
        [title_artist, status]
        + lines
        + header_artists
        + subheader_artists
        + [cell for row in row_artists for cell in row]
    )

# =========================
# Save
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

anim.save(GIF_PATH, writer=PillowWriter(fps=FPS))

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")